# ARC2 TRM AdamAtan2 compatibility smoke

Private, frozen one-task/one-step runtime diagnostic. It does not measure accuracy and cannot authorize sealed-holdout access. The only production-code change is the AdamAtan2 constructor bootstrap learning rate; the existing per-step warmup scheduler is unchanged.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

task_id = 'bc1d5164'
repair_id = "adam_atan2_bootstrap_base_lr_v1"
expected_runtime_sha256 = '17d8757c874f14d20aee0368054137874e8df16f2dd8f0fe9e521d5aff2c9e82'
started_at = time.time()

competition_root = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
training_path = competition_root / "arc-agi_training_challenges.json"
all_tasks = json.loads(training_path.read_text(encoding="utf-8"))
if task_id not in all_tasks:
    raise KeyError(f"frozen smoke task {task_id} missing from official training data")
challenge_path = Path("/kaggle/working/trm_smoke_challenge.json")
challenge_path.write_text(json.dumps({task_id: all_tasks[task_id]}), encoding="utf-8")

matches = sorted(Path("/kaggle/input").rglob("trm_runtime.py"))
if not matches:
    raise FileNotFoundError("attached ARC proof-search/TRM source is missing")
attached_source = matches[0].parent
trm_source = Path("/kaggle/working/hybrid_trm_source")
if trm_source.exists():
    shutil.rmtree(trm_source)
shutil.copytree(attached_source, trm_source)
sys.path.insert(0, str(trm_source))
from bootstrap_runtime import restore_archived_directories
restore_archived_directories(trm_source)

patched_runtime = '#!/usr/bin/env python3\n"""Offline ARC Prize 2026 runner for the MIT Tiny Recursive Model.\n\nThe runtime adapts the public TRM training entry point for Kaggle\'s offline\nenvironment, test-time trains on augmented demonstrations, and emits the\nstandard two-attempt submission.  The pretrained checkpoint is supplied as a\nseparate public Kaggle input and is never bundled into this source archive.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\n\nCHALLENGE_NAMES = (\n    "arc-agi-2_test_challenges.json",\n    "arc-agi_test_challenges.json",\n    "test_challenges.json",\n)\n\n\ndef find_challenge(input_root: Path) -> Path:\n    explicit = os.environ.get("ARC_CHALLENGE_FILE")\n    if explicit:\n        path = Path(explicit)\n        if path.is_file():\n            return path\n        raise FileNotFoundError(path)\n\n    for name in CHALLENGE_NAMES:\n        matches = sorted(input_root.rglob(name))\n        if matches:\n            return matches[0]\n    for path in sorted(input_root.rglob("*.json")):\n        if "test" not in path.name.lower() or "challenge" not in path.name.lower():\n            continue\n        try:\n            data = json.loads(path.read_text(encoding="utf-8"))\n        except (OSError, json.JSONDecodeError):\n            continue\n        if isinstance(data, dict) and data and all(\n            isinstance(task, dict) and "train" in task and "test" in task\n            for task in data.values()\n        ):\n            return path\n    raise FileNotFoundError("ARC-AGI-2 test challenge JSON not found")\n\n\ndef find_checkpoint(input_root: Path) -> Path:\n    explicit = os.environ.get("TRM_CHECKPOINT")\n    if explicit:\n        path = Path(explicit)\n        if path.is_file():\n            return path\n        raise FileNotFoundError(path)\n\n    preferred = ("step_220708", "step_723914")\n    for name in preferred:\n        matches = sorted(input_root.rglob(name))\n        if matches:\n            return matches[0]\n    matches = sorted(input_root.rglob("step_*"))\n    files = [path for path in matches if path.is_file()]\n    if files:\n        return files[-1]\n    raise FileNotFoundError(\n        "TRM checkpoint not found. Attach cpmpml/arc-prize-trm-031 or set TRM_CHECKPOINT."\n    )\n\n\ndef validate_tasks(path: Path) -> int:\n    tasks = json.loads(path.read_text(encoding="utf-8"))\n    if not isinstance(tasks, dict) or not tasks:\n        raise ValueError("challenge file must contain a non-empty task object")\n    for task_id, task in tasks.items():\n        if not isinstance(task, dict) or not task.get("train") or not task.get("test"):\n            raise ValueError(f"invalid task {task_id}")\n    return len(tasks)\n\n\ndef patch_trm_source(source: Path, runtime: Path) -> Path:\n    if runtime.exists():\n        shutil.rmtree(runtime)\n    shutil.copytree(source, runtime, ignore=shutil.ignore_patterns(".git", "__pycache__"))\n    entry = runtime / "pretrain.py"\n    text = entry.read_text(encoding="utf-8")\n\n    required = {\n        "import wandb": "# wandb disabled for the offline competition runtime",\n        "from adam_atan2 import AdamATan2": (\n            "from adam_atan2_pytorch import AdamAtan2"\n        ),\n        "AdamATan2(": "AdamAtan2(",\n        "if config.checkpoint_path is None or wandb.run is None:": (\n            "if config.checkpoint_path is None:"\n        ),\n        "wandb.run.log_code(config.checkpoint_path)": (\n            "print(f\'Runtime source saved in {config.checkpoint_path}\')"\n        ),\n        "wandb.finish()": "pass",\n    }\n    for old, new in required.items():\n        if old not in text:\n            raise RuntimeError(f"TRM patch anchor missing: {old}")\n        text = text.replace(old, new)\n\n    # adam-atan2-pytorch 0.2.4 rejects lr=0 at construction time, while the\n    # upstream TRM initializes Adam at zero and assigns the scheduled value\n    # immediately before every optimizer step.  Bootstrap Adam with the same\n    # declared base LR; the existing per-step schedule remains authoritative.\n    adam_zero_lr = re.compile(\n        r"(AdamAtan2\\(\\s*\\n\\s*model\\.parameters\\(\\),\\s*\\n\\s*)"\n        r"lr=0,(\\s*# Needs to be set by scheduler)"\n    )\n    text, adam_bootstrap_replacements = adam_zero_lr.subn(\n        r"\\1lr=config.lr,\\2",\n        text,\n    )\n    if adam_bootstrap_replacements != 2:\n        raise RuntimeError(\n            "expected two AdamAtan2 zero-LR bootstrap anchors, found "\n            f"{adam_bootstrap_replacements}"\n        )\n\n    lines = []\n    for line in text.splitlines():\n        stripped = line.strip()\n        if stripped.startswith("wandb.init("):\n            lines.append(f"{line[:len(line) - len(line.lstrip())]}print(\'offline TRM run\')")\n        elif stripped.startswith("wandb.log("):\n            lines.append(line.replace("wandb.log(", "print("))\n        else:\n            lines.append(line)\n    entry.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    return entry\n\n\ndef build_dataset(runtime: Path, challenge: Path, data_root: Path, num_aug: int) -> Path:\n    raw_root = data_root / "raw"\n    dataset_root = data_root / "arc2test"\n    raw_root.mkdir(parents=True, exist_ok=True)\n    copied = raw_root / "arc-agi_test_challenges.json"\n    shutil.copy2(challenge, copied)\n\n    command = [\n        sys.executable,\n        "-m",\n        "dataset.build_arc_dataset",\n        "--input-file-prefix",\n        str(raw_root / "arc-agi"),\n        "--output-dir",\n        str(dataset_root),\n        "--subsets",\n        "test",\n        "--test-set-name",\n        "test",\n        "--num-aug",\n        str(num_aug),\n    ]\n    subprocess.run(command, cwd=runtime, check=True)\n    return dataset_root\n\n\ndef training_command(\n    entry: Path,\n    dataset_root: Path,\n    checkpoint: Path,\n    output_root: Path,\n    epochs: int,\n    eval_interval: int,\n    global_batch_size: int,\n    learning_rate: float,\n    warmup_steps: int,\n    world_size: int,\n) -> list[str]:\n    return [\n        "torchrun",\n        "--standalone",\n        "--nnodes=1",\n        f"--nproc-per-node={world_size}",\n        "--rdzv_backend=c10d",\n        "--rdzv_endpoint=localhost:0",\n        str(entry),\n        "arch=trm",\n        f"data_paths=[{dataset_root}]",\n        "arch.L_layers=2",\n        "arch.H_cycles=4",\n        "arch.L_cycles=4",\n        "arch.halt_max_steps=10",\n        "freeze_weights=False",\n        f"+load_checkpoint={checkpoint}",\n        f"+checkpoint_path={output_root}",\n        f"eval_interval={eval_interval}",\n        f"epochs={epochs}",\n        f"global_batch_size={global_batch_size}",\n        "ema=True",\n        f"lr_warmup_steps={warmup_steps}",\n        f"lr={learning_rate}",\n    ]\n\n\ndef validate_submission(path: Path, challenge: Path) -> None:\n    tasks = json.loads(challenge.read_text(encoding="utf-8"))\n    submission = json.loads(path.read_text(encoding="utf-8"))\n    if set(submission) != set(tasks):\n        raise ValueError("submission task IDs do not match the challenge")\n    for task_id, task in tasks.items():\n        rows = submission[task_id]\n        if len(rows) != len(task["test"]):\n            raise ValueError(f"wrong output count for {task_id}")\n        for row in rows:\n            if set(row) != {"attempt_1", "attempt_2"}:\n                raise ValueError(f"wrong attempt keys for {task_id}")\n            for grid in row.values():\n                if not isinstance(grid, list) or not grid or not grid[0]:\n                    raise ValueError(f"empty grid for {task_id}")\n                if len(grid) > 30 or len(grid[0]) > 30:\n                    raise ValueError(f"oversized grid for {task_id}")\n                if any(len(line) != len(grid[0]) for line in grid):\n                    raise ValueError(f"non-rectangular grid for {task_id}")\n                if any(type(cell) is not int or not 0 <= cell <= 9 for line in grid for cell in line):\n                    raise ValueError(f"invalid color for {task_id}")\n\n\ndef latest_submission(output_root: Path) -> Path:\n    matches = list(output_root.rglob("submission.json"))\n    if not matches:\n        raise FileNotFoundError(f"TRM produced no submission below {output_root}")\n    return max(matches, key=lambda path: path.stat().st_mtime_ns)\n\n\ndef stepped_submissions(output_root: Path) -> list[tuple[int, Path]]:\n    """Return evaluator submissions ordered by their training step."""\n    found: list[tuple[int, Path]] = []\n    for path in output_root.rglob("submission.json"):\n        match = re.search(r"_step_(\\d+)$", path.parent.name)\n        if match:\n            found.append((int(match.group(1)), path))\n    return sorted(found)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dry-run", action="store_true")\n    args = parser.parse_args()\n\n    kaggle = Path("/kaggle")\n    input_root = Path(os.environ.get("ARC_INPUT_ROOT", kaggle / "input"))\n    working = Path(os.environ.get("ARC_WORKING_ROOT", kaggle / "working"))\n    source_root = Path(__file__).resolve().parent\n    trm_source = source_root / "third_party" / "TinyRecursiveModels"\n    if not (trm_source / "pretrain.py").is_file():\n        raise FileNotFoundError(f"bundled TRM source not found at {trm_source}")\n\n    challenge = find_challenge(input_root)\n    checkpoint = find_checkpoint(input_root)\n    task_count = validate_tasks(challenge)\n    epochs = int(os.environ.get("TRM_EPOCHS", "4000"))\n    eval_interval = int(os.environ.get("TRM_EVAL_INTERVAL", str(epochs)))\n    world_size = int(os.environ.get("TRM_WORLD_SIZE", "4"))\n    global_batch_size = int(os.environ.get("TRM_GLOBAL_BATCH_SIZE", "128"))\n    learning_rate = float(os.environ.get("TRM_LR", "0.0001"))\n    warmup_steps = int(os.environ.get("TRM_WARMUP_STEPS", "200"))\n    num_aug = int(os.environ.get("TRM_NUM_AUG", "128"))\n    if (\n        epochs <= 0\n        or eval_interval <= 0\n        or world_size <= 0\n        or global_batch_size <= 0\n        or learning_rate <= 0\n        or warmup_steps < 0\n        or num_aug <= 0\n    ):\n        raise ValueError(\n            "TRM_EPOCHS, TRM_EVAL_INTERVAL, TRM_WORLD_SIZE, "\n            "TRM_GLOBAL_BATCH_SIZE, TRM_LR, and TRM_NUM_AUG must be positive; "\n            "TRM_WARMUP_STEPS must be nonnegative"\n        )\n    if epochs % eval_interval:\n        raise ValueError("TRM_EVAL_INTERVAL must divide TRM_EPOCHS")\n    if global_batch_size % world_size:\n        raise ValueError("TRM_GLOBAL_BATCH_SIZE must be divisible by TRM_WORLD_SIZE")\n\n    runtime = working / "trm_runtime"\n    entry = patch_trm_source(trm_source, runtime)\n    print(f"TRM tasks: {task_count}; checkpoint: {checkpoint}")\n    if args.dry_run:\n        dataset_root = working / "trm_data" / "arc2test"\n        command = training_command(\n            entry,\n            dataset_root,\n            checkpoint,\n            working / "trm_output",\n            epochs,\n            eval_interval,\n            global_batch_size,\n            learning_rate,\n            warmup_steps,\n            world_size,\n        )\n        print("DRY RUN:", " ".join(map(str, command)))\n        return 0\n\n    data_root = working / "trm_data"\n    dataset_root = build_dataset(runtime, challenge, data_root, num_aug)\n    output_root = working / "trm_output"\n    command = training_command(\n        entry,\n        dataset_root,\n        checkpoint,\n        output_root,\n        epochs,\n        eval_interval,\n        global_batch_size,\n        learning_rate,\n        warmup_steps,\n        world_size,\n    )\n    environment = os.environ.copy()\n    environment.setdefault("OMP_NUM_THREADS", "8")\n    subprocess.run(command, cwd=runtime, env=environment, check=True)\n\n    generated = latest_submission(output_root)\n    destination = working / "trm_submission.json"\n    shutil.copy2(generated, destination)\n    validate_submission(destination, challenge)\n\n    stepped = stepped_submissions(output_root)\n    if len(stepped) >= 2:\n        early_step, early_path = stepped[0]\n        final_step, final_path = stepped[-1]\n        early_destination = working / "trm_submission_early.json"\n        final_destination = working / "trm_submission_final.json"\n        shutil.copy2(early_path, early_destination)\n        shutil.copy2(final_path, final_destination)\n        validate_submission(early_destination, challenge)\n        validate_submission(final_destination, challenge)\n        (working / "trm-agreement-checkpoints.json").write_text(\n            json.dumps(\n                {\n                    "policy": "agreement_between_early_and_final_rank1",\n                    "early_step": early_step,\n                    "final_step": final_step,\n                    "epochs": epochs,\n                    "eval_interval": eval_interval,\n                    "global_batch_size": global_batch_size,\n                    "learning_rate": learning_rate,\n                    "warmup_steps": warmup_steps,\n                },\n                indent=2,\n                sort_keys=True,\n            )\n            + "\\n",\n            encoding="utf-8",\n        )\n        print(\n            f"Exported agreement checkpoints at steps {early_step} and {final_step}"\n        )\n    print(f"Wrote and validated {destination}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
runtime_destination = trm_source / "trm_runtime.py"
runtime_destination.write_text(patched_runtime, encoding="utf-8")
actual_runtime_sha256 = hashlib.sha256(runtime_destination.read_bytes()).hexdigest()
assert actual_runtime_sha256 == expected_runtime_sha256

packages = [
    "hydra-core==1.3.2", "omegaconf==2.3.0",
    "adam_atan2_pytorch==0.2.4", "argdantic==1.3.3", "coolname==2.2.0",
]
subprocess.run([
    sys.executable, "-m", "pip", "install", "--no-index",
    f"--find-links={trm_source / 'wheels'}", *packages,
], check=True)

env = os.environ.copy()
env.update({
    "ARC_CHALLENGE_FILE": str(challenge_path),
    "ARC_WORKING_ROOT": "/kaggle/working",
    "CUDA_VISIBLE_DEVICES": "0",
    "TRM_WORLD_SIZE": "1",
    "TRM_EPOCHS": "1",
    "TRM_EVAL_INTERVAL": "1",
    "TRM_GLOBAL_BATCH_SIZE": "112",
    "TRM_LR": "0.0000875",
    "TRM_WARMUP_STEPS": "229",
    "TRM_NUM_AUG": "128",
    "OMP_NUM_THREADS": "4",
    "PYTHONNOUSERSITE": "1",
})
polluted_root = "/kaggle/usr/lib/notebooks/sorokin/pip_install_unsloth_flash_patch"
env["PYTHONPATH"] = os.pathsep.join(
    value for value in env.get("PYTHONPATH", "").split(os.pathsep)
    if value and polluted_root not in value
)

log_path = Path("/kaggle/working/trm-smoke.log")
with log_path.open("w", encoding="utf-8", buffering=1) as log_handle:
    result = subprocess.run(
        [sys.executable, str(runtime_destination)],
        env=env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
    )

submission_path = Path("/kaggle/working/trm_submission.json")
submission_valid = False
submission_sha256 = None
if result.returncode == 0 and submission_path.is_file():
    from trm_runtime import validate_submission
    validate_submission(submission_path, challenge_path)
    submission_valid = True
    submission_sha256 = hashlib.sha256(submission_path.read_bytes()).hexdigest()

log_text = log_path.read_text(encoding="utf-8", errors="replace")
step_one_checkpoint = Path("/kaggle/working/trm_output/step_1")
guards = {
    "runtime_returncode_zero": result.returncode == 0,
    "checkpoint_loaded": "Loading checkpoint" in log_text,
    "optimizer_constructed": "AssertionError" not in log_text,
    "first_step_checkpoint_saved": step_one_checkpoint.is_file(),
    "submission_exists": submission_path.is_file(),
    "submission_schema_valid": submission_valid,
}
receipt = {
    "purpose": "runtime compatibility smoke; not an accuracy experiment",
    "repair_id": repair_id,
    "single_change": "AdamAtan2 constructor bootstrap lr: 0 -> config.lr; scheduled per-step lr unchanged",
    "task_id": task_id,
    "task_count": 1,
    "epochs": 1,
    "expected_optimizer_steps": 1,
    "runtime_sha256": actual_runtime_sha256,
    "challenge_sha256": hashlib.sha256(challenge_path.read_bytes()).hexdigest(),
    "submission_sha256": submission_sha256,
    "elapsed_seconds": time.time() - started_at,
    "returncode": result.returncode,
    "guards": guards,
    "all_guards_passed": all(guards.values()),
}
receipt_path = Path("/kaggle/working/trm-smoke-receipt.json")
receipt_path.write_text(json.dumps(receipt, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(receipt, indent=2, sort_keys=True))
if not receipt["all_guards_passed"]:
    print("*** TRM SMOKE LOG TAIL ***")
    print("\n".join(log_text.splitlines()[-100:]))
    raise SystemExit(1)
